# Codebase Onboarding Agent —— 代码库入门问答智能体

用 Python AST 索引、BM25 与向量检索找源码证据，并提供可选的 HelloAgents ReAct 问答演示。

| Part | 内容 | 验证范围 |
|---|---|---|
| 1 | AST 切块与符号索引 | 文件、符号、起始行坐标 |
| 2 | BM25 + dense + 加权 RRF | 检索排序 |
| 3 | 四个只读工具 + ReAct 演示 | 引用 ID 存在且在本轮工具结果中出现；不验证断言是否被证据支持 |
| 4 | 三个主臂 + 三个诊断臂 | 确定性证据检索，不运行 Part 3 的 Agent 或回答生成 |

它是课程项目中检索与评测方法的教学实现。原项目的服务架构、LLM 最终引用评测和三轮重复没有迁入本 notebook。
方法说明见配套投稿 Extra14；两个 PR 可独立运行与阅读，合并状态由各自 PR 页面说明。


## 运行须知

- Python 3.12；运行前在本项目目录安装 `requirements.txt`。
- Part 1/2/4 不需要 API Key；首次运行需要联网下载 Requests 语料和 embedding 模型。
- Part 3 默认跳过。只有主动设置 `RUN_AGENT_DEMO=1` 并配置 key 才会调用 LLM。
- 参考环境使用 CPU、固定模型 revision 和依赖版本。其他硬件或数值库仍可能影响边界排序，不承诺跨环境逐位一致。
- 先运行 `python -m unittest -v` 检查评分器。完整结果写入 `results/h1_report.json`，提交的参考产物在 `reference/h1_report.json`。


---

## 0. 环境准备

In [ ]:
# 首次运行取消注释
# !pip install -q -r requirements.txt

In [ ]:
import ast, io, json, os, re, math, hashlib, tarfile, urllib.request, platform
import importlib.metadata
from evaluation import parse_chunk, anchor_match, validate_questions, ranked_metrics, decide
from evaluation import verify_citations as check_citation_ids
from collections import defaultdict, Counter
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional

BASE = Path(".").resolve()
DATA = BASE / "data"
CORPUS_DIR = BASE / "corpus"
print("工作目录:", BASE)

---

# Part 1 · 结构化索引

## 1.1 先把语料钉死

评测的第一条纪律是**冻结被测对象**。语料用 `psf/requests` 的一个具体 commit，不是 `main`——否则今天和下周跑出来的数字没法比。

In [ ]:
CORPUS_REPO = "psf/requests"
CORPUS_COMMIT = "414f0513c33883adf6f2b46901d4f0b38a455851"   # 固定 commit，不用 main

def fetch_corpus() -> Path:
    src_dir = CORPUS_DIR / f"requests-{CORPUS_COMMIT}" / "src" / "requests"
    if src_dir.exists():
        return src_dir
    CORPUS_DIR.mkdir(exist_ok=True)
    url = f"https://codeload.github.com/{CORPUS_REPO}/tar.gz/{CORPUS_COMMIT}"
    print(f"下载语料 {CORPUS_REPO}@{CORPUS_COMMIT[:8]} ...")
    with urllib.request.urlopen(url) as r:
        buf = io.BytesIO(r.read())
    with tarfile.open(fileobj=buf, mode="r:gz") as tar:
        tar.extractall(CORPUS_DIR, filter="data")
    return src_dir

CORPUS_SRC = fetch_corpus()
py_files = sorted(p for p in CORPUS_SRC.rglob("*.py"))
print(f"语料就绪: {CORPUS_SRC.relative_to(BASE)}")
print(f"Python 文件 {len(py_files)} 个")

## 1.2 按语法边界切块，而不是按固定字符数

固定长度切块会把一个函数从中间劈开，检索命中了也没法给出"这段逻辑在哪个符号里"。

本项目按 Python AST 的结构边界切：模块 docstring、顶层函数、类、类里的方法。**每个 chunk 都带一组稳定坐标 `file_path + symbol_name + start_line`**，后面 BM25、向量检索、引用定位和评测判分全都用这一份坐标，不会各说各话。

注意 `class` chunk 和它的 `method` chunk 是**有意重叠**的：问"这个类负责什么"要看整个类，问"这段逻辑怎么写的"要看具体方法。

In [ ]:
@dataclass
class CodeChunk:
    chunk_id: str
    file_path: str          # 相对语料根，如 "auth.py"
    chunk_type: str         # module_doc | class | function | method
    symbol_name: str
    parent_symbol: Optional[str]
    signature: str
    start_line: int
    end_line: int
    imports: list = field(default_factory=list)
    content: str = ""

    @property
    def label(self) -> str:
        if self.parent_symbol:
            return f"{self.file_path}::{self.parent_symbol}.{self.symbol_name}"
        return f"{self.file_path}::{self.symbol_name}"


def _signature(node) -> str:
    try:
        args = ast.unparse(node.args)
    except Exception:
        args = "..."
    prefix = "async def " if isinstance(node, ast.AsyncFunctionDef) else "def "
    return f"{prefix}{node.name}({args})"


def _segment(lines, node) -> str:
    end = getattr(node, "end_lineno", node.lineno)
    return "\n".join(lines[node.lineno - 1 : end])


def chunk_file(path: Path, rel: str) -> list:
    src = path.read_text(encoding="utf-8", errors="replace")
    try:
        tree = ast.parse(src)
    except SyntaxError:
        # 一个坏文件不该拖垮整个仓库的索引
        print(f"  ⚠️ 跳过（语法错误）: {rel}")
        return []

    lines = src.splitlines()
    imports = []
    for n in ast.walk(tree):
        if isinstance(n, ast.Import):
            imports += [a.name for a in n.names]
        elif isinstance(n, ast.ImportFrom) and n.module:
            imports.append(n.module)
    imports = sorted(set(imports))

    out, seq = [], 0

    def emit(**kw):
        nonlocal seq
        seq += 1
        out.append(CodeChunk(chunk_id=f"{rel}#{seq}", file_path=rel,
                             imports=imports, **kw))

    doc = ast.get_docstring(tree)
    if doc:
        emit(chunk_type="module_doc", symbol_name=Path(rel).stem,
             parent_symbol=None, signature=f"module {rel}",
             start_line=tree.body[0].lineno, end_line=tree.body[0].end_lineno,
             content=_segment(lines, tree.body[0]))

    for node in tree.body:
        if isinstance(node, ast.ClassDef):
            bases = ", ".join(getattr(b, "id", "?") for b in node.bases)
            emit(chunk_type="class", symbol_name=node.name, parent_symbol=None,
                 signature=f"class {node.name}({bases})",
                 start_line=node.lineno, end_line=node.end_lineno,
                 content=_segment(lines, node))
            for sub in node.body:
                if isinstance(sub, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    emit(chunk_type="method", symbol_name=sub.name,
                         parent_symbol=node.name, signature=_signature(sub),
                         start_line=sub.lineno, end_line=sub.end_lineno,
                         content=_segment(lines, sub))
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            emit(chunk_type="function", symbol_name=node.name, parent_symbol=None,
                 signature=_signature(node),
                 start_line=node.lineno, end_line=node.end_lineno,
                 content=_segment(lines, node))
    return out


CHUNKS = []
for p in py_files:
    CHUNKS += chunk_file(p, str(p.relative_to(CORPUS_SRC)))

BY_ID = {c.chunk_id: c for c in CHUNKS}
print(f"共 {len(CHUNKS)} 个 chunk")
print("类型分布:", dict(Counter(c.chunk_type for c in CHUNKS)))

In [ ]:
# 看一个具体的 chunk：坐标齐全，能直接点回源码
demo = next(c for c in CHUNKS if c.symbol_name == "prepare_auth")
print(f"label     : {demo.label}")
print(f"type      : {demo.chunk_type}")
print(f"signature : {demo.signature}")
print(f"lines     : {demo.start_line}–{demo.end_line}")
print("-" * 60)
print("\n".join(demo.content.splitlines()[:12]))

---

# Part 2 · 混合检索

## 2.1 为什么保留两路

符号名和配置键适合尝试词面检索，行为描述适合尝试语义召回。具体效果取决于模型、语料和问法；
保留两条可单独运行的路径，才能测量融合是否带来收益。


In [ ]:
from rank_bm25 import BM25Okapi

_CAMEL = re.compile(r"(?<=[a-z0-9])(?=[A-Z])")

def tokenize(text: str) -> list:
    '''代码检索的分词：先按非字母数字切，再拆 camelCase，最后小写。
    `prepareAuth` / `prepare_auth` / `PrepareAuth` 会落到同一批 token 上。'''
    parts = re.split(r"[^A-Za-z0-9]+", text)
    out = []
    for p in parts:
        if not p:
            continue
        out += [s.lower() for s in _CAMEL.split(p) if s]
    return out


def bm25_text(c: CodeChunk) -> str:
    return f"{c.file_path} {c.parent_symbol or ''} {c.symbol_name} {c.signature}\n{c.content}"

BM25_CORPUS = [tokenize(bm25_text(c)) for c in CHUNKS]
BM25 = BM25Okapi(BM25_CORPUS)
print(f"BM25 索引就绪，{len(BM25_CORPUS)} 篇文档")

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Pin the model snapshot; package versions are in requirements.txt.
EMBED_MODEL = "flax-sentence-embeddings/st-codesearch-distilroberta-base"
EMBED_REVISION = "23d22ea4191fc9d006833d3b62694949f1ffebd1"

def embed_text(c: CodeChunk) -> str:
    return f"{c.file_path} {c.signature}\n{c.content[:1500]}"

encoder = SentenceTransformer(EMBED_MODEL, revision=EMBED_REVISION, device="cpu")
DENSE = encoder.encode([embed_text(c) for c in CHUNKS],
                       batch_size=64, normalize_embeddings=True,
                       show_progress_bar=True)
print("向量矩阵:", DENSE.shape)


## 2.2 加权 RRF

BM25 与余弦相似度的尺度不同，融合原始分数需要校准。RRF 用名次融合，避免直接比较两路原始分数，代价是丢失分数幅度信息。

本例使用 `K=60`、`dense:sparse=2:1`，属于已有开发期选择，没有经过独立开发集验证。
换语料需要重新评估，不能把这些参数当成通用最优值。


In [ ]:
RRF_K = 60
W_DENSE, W_SPARSE = 2.0, 1.0

def bm25_rank(query: str, n: int = 50) -> list:
    scores = BM25.get_scores(tokenize(query))
    idx = np.argsort(-scores, kind="stable")[:n]
    return [CHUNKS[i].chunk_id for i in idx if scores[i] > 0]

def dense_rank(query: str, n: int = 50) -> list:
    qv = encoder.encode([query], normalize_embeddings=True)[0]
    scores = DENSE @ qv
    idx = np.argsort(-scores, kind="stable")[:n]
    return [CHUNKS[i].chunk_id for i in idx]

def weighted_rrf(rank_lists, weights, k=RRF_K) -> list:
    acc = defaultdict(float)
    for ranked, w in zip(rank_lists, weights):
        for rank, cid in enumerate(ranked, start=1):
            acc[cid] += w / (k + rank)
    return [cid for cid, _ in sorted(acc.items(), key=lambda kv: -kv[1])]

def hybrid_rank(query: str, n: int = 50) -> list:
    return weighted_rrf([dense_rank(query, n), bm25_rank(query, n)],
                        [W_DENSE, W_SPARSE])

In [ ]:
q = "How is basic auth attached to a prepared request?"
print("BM25 前 5:")
for cid in bm25_rank(q)[:5]:
    print("   ", BY_ID[cid].label)
print("\nDense 前 5:")
for cid in dense_rank(q)[:5]:
    print("   ", BY_ID[cid].label)
print("\n加权 RRF 前 5:")
for cid in hybrid_rank(q)[:5]:
    print("   ", BY_ID[cid].label)

---

# Part 3 · Agent、工具与引用 ID 校验

跨文件问题可能需要继续查看源码。下面提供搜索、大纲、读符号、查潜在调用方四个只读工具。
调用名匹配无法解析所有动态分派，例如 `auth(self)` 不会自动解析为某个类的 `__call__`；Agent 仍需结合源码判断。


In [ ]:
# Dedent methods before parsing. Parse failures are errors, not missing edges.
CALL_SITES = defaultdict(set)
DEFINED_BY = defaultdict(list)
CHUNK_TREES = {}
for c in CHUNKS:
    if c.chunk_type not in ("class", "function", "method"):
        continue
    DEFINED_BY[c.symbol_name].append(c.chunk_id)
    CHUNK_TREES[c.chunk_id] = parse_chunk(c.content)
    for node in ast.walk(CHUNK_TREES[c.chunk_id]):
        if isinstance(node, ast.Call):
            name = getattr(node.func, "id", None) or getattr(node.func, "attr", None)
            if name:
                CALL_SITES[name].add(c.chunk_id)
print(f"成功解析 {len(CHUNK_TREES)} 个类/函数/方法 chunk")
print("调用名匹配仅生成候选，不做接收者类型推断，可能漏报或误报。")


In [ ]:
OBSERVED_IDS = set()

def source_url(c):
    return (f"https://github.com/{CORPUS_REPO}/blob/{CORPUS_COMMIT}/src/requests/"
            f"{c.file_path}#L{c.start_line}-L{c.end_line}")

def tool_search_code(query: str) -> str:
    ids = hybrid_rank(query)[:5]
    OBSERVED_IDS.update(ids)
    return "\n\n".join(f"[{cid}] {BY_ID[cid].label}\n{BY_ID[cid].content[:800]}"
                        for cid in ids) or "no match"

def tool_file_outline(file_path: str) -> str:
    items = sorted((c for c in CHUNKS if c.file_path == file_path), key=lambda c: c.start_line)
    OBSERVED_IDS.update(c.chunk_id for c in items)
    return "\n".join(f"[{c.chunk_id}] L{c.start_line} {c.label}" for c in items) or "file not found"

def tool_read_symbol(symbol: str) -> str:
    ids = ([symbol] if symbol in BY_ID else
           [c.chunk_id for c in CHUNKS if c.label == symbol] or DEFINED_BY.get(symbol, []))
    if not ids:
        return f"symbol not found: {symbol}"
    if len(ids) != 1:
        return "Ambiguous symbol; use an evidence ID:\n" + "\n".join(
            f"{i}: {BY_ID[i].label}" for i in ids)
    c = BY_ID[ids[0]]
    OBSERVED_IDS.add(c.chunk_id)
    return f"[{c.chunk_id}] {c.label}\n{c.content}"

def tool_find_callers(symbol: str) -> str:
    ids = sorted(CALL_SITES.get(symbol, []))[:8]
    OBSERVED_IDS.update(ids)
    return "\n".join(f"[{i}] {BY_ID[i].label}" for i in ids) or "no caller found"

print(tool_find_callers("_basic_auth_str"))


## 3.1 引用 ID 校验

`citation_id_validity` 是过滤前不同引用 ID 中通过校验的比例，重复同一个 ID 不增加分母或分子；零引用记 0。
Agent 演示还要求该 ID 在本轮工具结果中出现，删除不存在或未观察到的 ID，并提供固定 commit 的源码链接。

这只是引用位置校验，**不是 groundedness**。真实 ID 仍可能被挂在错误断言后面；删除无效标记也不会纠正正文。
这里不评价答案的语义正确性，这个比例也不进入 Part 4 的检索 composite。


In [ ]:
def verify_citations(answer: str, allowed_ids=None):
    return check_citation_ids(answer, BY_ID, allowed_ids)

real_id = next(c.chunk_id for c in CHUNKS if c.symbol_name == "_basic_auth_str")
r = verify_citations(f"Example [{real_id}] and invalid [nonexistent.py#9].")
print(r)
# A real ID can still accompany a false claim. This ratio is not groundedness.


## 3.2 可选的 HelloAgents 演示

显式启用后会调用配置的 LLM，调用错误直接暴露。自动评测始终跳过这一段。
`agent_demo.py` 通过 0.2.7 的 `custom_prompt` 传入规则；单独传 `system_prompt` 不会进入该版本的 ReAct 循环提示词。


In [ ]:
from agent_demo import build_agent, configured_llm
from dotenv import load_dotenv
load_dotenv(BASE / ".env")
agent = None
api_key = os.getenv("LLM_API_KEY", "").strip()
if os.getenv("RUN_AGENT_DEMO") == "1" and api_key and api_key != "your_key_here":
    agent = build_agent(configured_llm(), {
        "search_code": tool_search_code, "file_outline": tool_file_outline,
        "read_symbol": tool_read_symbol, "find_callers": tool_find_callers,
    })
    print("Agent 就绪")
else:
    print("跳过可选演示：配置 key 和 RUN_AGENT_DEMO=1 后启用。Part 4 无需 LLM。")


In [ ]:
if agent is not None:
    question = "When following a redirect to a different host, how does a Session decide whether to strip the Authorization header?"
    OBSERVED_IDS.clear()
    raw_answer = agent.run(question)

    checked = verify_citations(str(raw_answer), OBSERVED_IDS)
    print("\n" + "=" * 70)
    print(checked["answer"])
    print("=" * 70)
    print(f"引用（删除前）: {len(checked['cited_before_filter'])} 条")
    print(f"验证通过      : {len(checked['verified'])} 条")
    print(f"引用 ID 有效率: {checked['citation_id_validity']:.2f}")
    for cid in checked["verified"]:
        c = BY_ID[cid]
        display(__import__("IPython").display.Markdown(f"- [{c.label}]({source_url(c)})"))
else:
    print("跳过（未配置 LLM）")

---

# Part 4 · 确定性证据检索评测

我们比较 dense、hybrid 和 hybrid 加一跳取证扩展。指标回答“前五条证据命中了哪些正确源码位置”，
**不测 ReAct 工具选择、最终回答或引用 ID 有效率**。只改检索路径，才可以解释这三个臂的差值。


## 4.1 一道题长什么样

问题附带 `file + symbol + start_line` 锚点。v2 协议精确匹配三个字段，类块不能代替其方法，
同名方法用起始行区分；每个 gold 只计一次，gold 顺序不影响分数。

- Recall@5：前五条结果覆盖的 gold 比例。
- MRR@5：前五条中第一个正确结果的倒数排名，未命中记 0。
- nDCG@5：按排名折扣累加不同 gold 的命中，除以理想排序得分。

这是较严格的证据定位指标：相关的大类块也可能得零，gold 未标到的合理证据不会获分。
启动时要求所有 gold 恰好解析到一个 chunk，避免题集失效却继续出分。


In [ ]:
QUESTIONS_PATH = DATA / "questions.jsonl"
QUESTIONS = [json.loads(l) for l in QUESTIONS_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]

validate_questions(QUESTIONS, CHUNKS)

# 题集 SHA：跑完对一次，确保中途没人（包括我自己）偷偷改题
QUESTIONS_SHA256 = hashlib.sha256(QUESTIONS_PATH.read_bytes()).hexdigest()

print(f"题集 {len(QUESTIONS)} 道，SHA256 {QUESTIONS_SHA256[:16]}…")
print("层级分布:", dict(Counter(q["taxonomy"] for q in QUESTIONS)))
print("来源分布:", dict(Counter(q["source"] for q in QUESTIONS)))
print()
sample = QUESTIONS[2]
print("示例题:", sample["question"])
for t in sample["gt_targets"]:
    print(f"   gold → {t['file']}::{t['symbol']}  L{t['start_line']}")

> 10 道题里有 4 道标为 `graph_reverse`，从已有调用关系反推，因此可能漏掉解析器看不到的链路。
> 它们集中在 L2/L3，来源和难度存在混淆；这份诊断集不能代表无偏的用户问题分布。


In [ ]:
# Protocol v2, exact anchors; see evaluation.py and test_evaluation.py.
def recall_at_k(ranked_ids, golds, k=5):
    return ranked_metrics(ranked_ids, golds, BY_ID, k)["recall@5"]

def mrr(ranked_ids, golds, k=5):
    return ranked_metrics(ranked_ids, golds, BY_ID, k)["mrr@5"]

def ndcg_at_k(ranked_ids, golds, k=5):
    return ranked_metrics(ranked_ids, golds, BY_ID, k)["ndcg@5"]


## 4.2 对照阶梯

| 臂 | 路径 |
|---|---|
| B2 | dense-only 的前五条证据 |
| B3 | BM25 + dense + 加权 RRF |
| B4 | B3 种子 + 一跳调用名扩展与显式符号跳转，再融合排序 |

三者共享索引、模型和评分器；没有 LLM 合成，因此 B2 是 dense 检索基线，B4 也不是完整 ReAct 系统。
`B4−B3` 衡量本配置下新增扩展策略的收益；`B4−B2` 同时包含混合检索收益。

扩展最多检查六个种子，按来源名次和向量相关度排序，14 步是该检索策略的操作上限。
调用解析不做接收者类型推断，同名定义全部作为候选，因此可能误报；一步也可能产生很多候选，步数不等于 token 或延迟预算。


In [ ]:
TOP_K = 5

def arm_B2(question: str) -> list:
    return dense_rank(question, 50)[:TOP_K]

def arm_B3(question: str) -> list:
    return hybrid_rank(question, 50)[:TOP_K]

POS = {c.chunk_id: i for i, c in enumerate(CHUNKS)}

def expand_seed(question, seed, budget=14):
    """One-hop expansion, shared by B4 and B4Qs. Same original-question jumps.

    One search, at most six seed inspections, then explicit-symbol lookups.
    Each inspection/lookup costs one step, including unresolved names.
    This is a deterministic retrieval policy, not the ReAct execution loop.
    """
    if budget < 1:
        raise ValueError("budget must include the initial search")
    steps, provenance = 1, {}
    for rank, cid in enumerate(seed[:6], 1):
        if steps >= budget:
            break
        steps += 1
        tree = CHUNK_TREES.get(cid)
        if tree is None:
            continue
        for node in ast.walk(tree):
            if isinstance(node, ast.Call):
                name = getattr(node.func, "id", None) or getattr(node.func, "attr", None)
                # Name-only resolution can be ambiguous: retain candidates, don't
                # silently pick the first class's method as the correct target.
                for target in DEFINED_BY.get(name, []):
                    provenance.setdefault(target, rank)
    for token in sorted(set(re.findall(r"[A-Za-z_][A-Za-z0-9_]{3,}", question))):
        if steps >= budget:
            break
        if token in DEFINED_BY:
            steps += 1
            for target in DEFINED_BY[token]:
                provenance[target] = 0
    qv = encoder.encode([question], normalize_embeddings=True)[0]
    followed = sorted(provenance, key=lambda cid: (
        provenance[cid], -float(DENSE[POS[cid]] @ qv), POS[cid]))
    return weighted_rrf([seed, followed], [3.0, 1.0])[:TOP_K]

def arm_B4(question: str, budget: int = 14) -> list:
    return expand_seed(question, hybrid_rank(question, 50), budget)

ARMS = {"B2": arm_B2, "B3": arm_B3, "B4": arm_B4}
print("进判据的对照臂：", list(ARMS))


### 查询改写诊断

确定性扩展包括同义词、符号表回填，保留原问题。它方便隔离查询输入的变化；效果需要在本题集上测量。
这些词表与 RRF 权重已有开发期调整，本题集是教学诊断集，不是独立 holdout。


In [ ]:
# 代码检索场景的词表。真实系统会用领域词典或 LLM 生成，这里手写以保证确定性。
CODE_SYNONYMS = {
    "build": ["make", "create", "construct", "prepare"],
    "create": ["make", "new", "init", "prepare"],
    "check": ["verify", "validate", "is", "has"],
    "decide": ["check", "verify", "should", "resolve"],
    "get": ["fetch", "read", "load", "resolve"],
    "set": ["update", "write", "apply"],
    "handle": ["process", "dispatch", "resolve", "rebuild"],
    "strip": ["remove", "delete", "clear", "pop", "rebuild"],
    "extract": ["parse", "read", "get"],
    "merge": ["update", "combine", "apply"],
    "store": ["save", "set", "update", "jar"],
    "follow": ["resolve", "redirect", "next"],
    "infer": ["get", "detect", "guess", "from"],
    "trace": ["call", "send", "request", "dispatch"],
    "path": ["flow", "chain", "send"],
    "encoding": ["charset", "decode", "encode"],
    "header": ["headers"],
    "redirect": ["redirects", "location", "resolve"],
    "auth": ["authorization", "authenticate", "credentials", "basic"],
    "authorization": ["auth", "credentials"],
    "cookie": ["cookies", "cookiejar", "jar"],
    "proxy": ["proxies", "bypass", "environment"],
    "cert": ["certificate", "ssl", "tls", "verify"],
    "certificate": ["cert", "ssl", "tls", "verify"],
    "body": ["data", "payload", "content"],
    "session": ["sessions"],
    "adapter": ["adapters", "transport"],
}

MAX_EXPAND = 12

def rewrite_query(question: str, max_expand: int = MAX_EXPAND):
    '''确定性查询改写，返回 (改写后的查询, 新增的词)。'''
    toks = set(tokenize(question))
    extra = []
    for t in sorted(toks):        # 必须排序：set 的迭代顺序随 PYTHONHASHSEED 变，
        extra += CODE_SYNONYMS.get(t, [])   # 不排序的话截断到 max_expand 时选中的词每次都不同
    for sym in DEFINED_BY:                       # 符号表回填
        if len(set(tokenize(sym)) & toks) >= 2:
            extra.append(sym)
    extra = [e for e in dict.fromkeys(extra) if e not in toks][:max_expand]
    return (question + " " + " ".join(extra)).strip(), extra

demo_q = "How is basic auth attached to a prepared request?"
print(demo_q)
print("扩展出：", rewrite_query(demo_q)[1])

B3Q 将扩展查询给两路；B3Qs 只改稀疏路。
B4Qs 在 B3Qs 的种子上调用与 B4 完全相同的扩展函数，显式符号跳转仍使用原问题，避免同时改动两个因素。


In [ ]:
def arm_B3Q(question: str) -> list:
    '''诊断臂：改写后的查询给两路。'''
    return hybrid_rank(rewrite_query(question)[0], 50)[:TOP_K]

def arm_B3Qs(question: str) -> list:
    '''诊断臂：只给稀疏臂改写，稠密臂用原问题。'''
    rq, _ = rewrite_query(question)
    return weighted_rrf([dense_rank(question, 50), bm25_rank(rq, 50)],
                        [W_DENSE, W_SPARSE])[:TOP_K]

def arm_B4Qs(question: str) -> list:
    '''诊断臂：改写(仅稀疏) + 一跳扩展，看两个增强叠不叠加。'''
    rq, _ = rewrite_query(question)
    seed = weighted_rrf([dense_rank(question, 50), bm25_rank(rq, 50)], [W_DENSE, W_SPARSE])
    return expand_seed(question, seed)

DIAGNOSTIC = {"B3Q": arm_B3Q, "B3Qs": arm_B3Qs, "B4Qs": arm_B4Qs}
print("诊断臂（不进判据）：", list(DIAGNOSTIC))


## 4.3 运行前固定判据

本次修复沿用既有四项比较与 0.05 门槛，不按重跑结果调整。诊断臂不参与判定。

**边界：这不是一次新的独立预注册验证。** 10 道题已经参与开发、配置比较和评分器修复；
这里只演示“先定义规则，再运行与报告”的流程。要得到确认性结论，需要提前冻结代码、题集和判据，并在未用于调试的数据上运行。


In [ ]:
DECISION_RULE = {
    "treatment": "B4",
    "controls": ["B2", "B3"],          # 两个对手都要打过
    "strata": ["L2", "L3"],            # 主判据聚焦跨文件和架构题；L1 作为辅助观察
    "min_margin": 0.05,                # 工程门槛，不是统计显著性阈值
    "composite": ["recall@5", "mrr@5", "ndcg@5"],   # 三项均值
}

# 2 层 × 2 个对手 = 4 项比较，任意一项不满足 → 整体 unsupported
N_COMPARISONS = len(DECISION_RULE["strata"]) * len(DECISION_RULE["controls"])

PROVENANCE = {
    "protocol": "strict-anchor-v2",
    "evaluation_scope": "deterministic evidence retrieval; no LLM answers",
    "python": platform.python_version(),
    "platform": platform.platform(),
    "packages": {p: importlib.metadata.version(p) for p in ("sentence-transformers", "torch", "transformers", "numpy", "rank-bm25", "pandas")},
    "evaluation_sha256": hashlib.sha256((BASE / "evaluation.py").read_bytes()).hexdigest(),
    "code_cells_sha256": hashlib.sha256("".join("".join(c["source"]) for c in json.loads((BASE / "main.ipynb").read_text())["cells"] if c["cell_type"] == "code").encode()).hexdigest(),
    "embed_revision": EMBED_REVISION,
    "device": "cpu",
    "expansion": {"budget": 14, "seed_count": 6, "rrf_weights": [3, 1]},
    "max_expand": MAX_EXPAND,
    "corpus": f"{CORPUS_REPO}@{CORPUS_COMMIT}",
    "n_files": len(py_files),
    "n_chunks": len(CHUNKS),
    "questions_sha256": QUESTIONS_SHA256,
    "embed_model": EMBED_MODEL,
    "rrf": {"K": RRF_K, "dense": W_DENSE, "sparse": W_SPARSE},
    "top_k": TOP_K,
    "decision_rule": DECISION_RULE,
}
print(json.dumps(PROVENANCE, indent=2, ensure_ascii=False))
print(f"\n共 {N_COMPARISONS} 项比较，全过才算 supported。")

In [ ]:
def run_suite() -> dict:
    per_q = []
    for q in QUESTIONS:
        row = {"id": q["id"], "taxonomy": q["taxonomy"], "source": q["source"]}
        for name, fn in {**ARMS, **DIAGNOSTIC}.items():
            ranked = fn(q["question"])
            row[name] = {
                "recall@5": recall_at_k(ranked, q["gt_targets"], TOP_K),
                "mrr@5": mrr(ranked, q["gt_targets"], TOP_K),
                "ndcg@5": ndcg_at_k(ranked, q["gt_targets"], TOP_K),
                "top": [{"id": c, "file": BY_ID[c].file_path, "symbol": BY_ID[c].symbol_name, "start_line": BY_ID[c].start_line} for c in ranked],
            }
        per_q.append(row)
    return {"per_question": per_q}

def composite(cell: dict) -> float:
    return sum(cell[m] for m in DECISION_RULE["composite"]) / len(DECISION_RULE["composite"])

RESULT = run_suite()
n_arms = len(ARMS) + len(DIAGNOSTIC)
print(f"完成 {len(QUESTIONS)} 题 × {n_arms} 臂 = {len(QUESTIONS) * n_arms} 次执行")

## 4.4 结果

In [ ]:
import pandas as pd

overall = []
for name in {**ARMS, **DIAGNOSTIC}:
    cells = [r[name] for r in RESULT["per_question"]]
    overall.append({
        "臂": name,
        "Recall@5": sum(c["recall@5"] for c in cells) / len(cells),
        "MRR@5":      sum(c["mrr@5"] for c in cells) / len(cells),
        "nDCG@5":   sum(c["ndcg@5"] for c in cells) / len(cells),
        "composite": sum(composite(c) for c in cells) / len(cells),
    })
df_overall = pd.DataFrame(overall).set_index("臂")
print("整体阶梯（10 题全体）")
display(df_overall.round(3))

In [ ]:
rows = []
for stratum in DECISION_RULE["strata"]:
    subset = [r for r in RESULT["per_question"] if r["taxonomy"] == stratum]
    if not subset:
        raise ValueError(f"Missing required stratum: {stratum}")
    row = {"层级": stratum, "题数": len(subset)}
    for name in {**ARMS, **DIAGNOSTIC}:
        row[name] = sum(composite(r[name]) for r in subset) / len(subset)
    for ctrl in DECISION_RULE["controls"]:
        row[f"B4−{ctrl}"] = row["B4"] - row[ctrl]
    rows.append(row)

df_strata = pd.DataFrame(rows).set_index("层级")
print("分层 composite 与 margin")
display(df_strata)

In [ ]:
scores = {r["层级"]: r for r in rows}
checks, verdict = decide(scores, DECISION_RULE)
passed = sum(c["passed"] for c in checks)
display(pd.DataFrame(checks))
print(f"H1 verdict: {verdict}, {passed}/{N_COMPARISONS}")


### 判据之后看诊断臂

诊断结果用于分析，不改变主判定；单个子组超过 0.05 不表示通过主判据或具有统计显著性。


In [ ]:
# 两个增强各自相对 B3 的孤立贡献
iso = []
for row in rows:
    st = row["层级"]
    iso.append({
        "层级": st,
        "改写(两路都改) B3Q−B3": round(row["B3Q"] - row["B3"], 4),
        "改写(仅稀疏) B3Qs−B3": round(row["B3Qs"] - row["B3"], 4),
        "一跳扩展 B4−B3": round(row["B4"] - row["B3"], 4),
        "两者叠加 B4Qs−B3": round(row["B4Qs"] - row["B3"], 4),
    })
display(pd.DataFrame(iso).set_index("层级"))

In [ ]:
# 扩展词数的敏感性。注意：这是在同一批题上扫的，见下面的说明
sens = []
for budget in (4, 8, 12, 20):
    tot = 0.0
    for q in QUESTIONS:
        rq = rewrite_query(q["question"], budget)[0]
        ranked = weighted_rrf([dense_rank(q["question"], 50), bm25_rank(rq, 50)],
                              [W_DENSE, W_SPARSE])[:TOP_K]
        tot += composite({m: fn(ranked, q["gt_targets"], TOP_K) for m, fn in
                          (("recall@5", recall_at_k), ("mrr@5", mrr), ("ndcg@5", ndcg_at_k))})
    sens.append({"max_expand": budget, "整体 composite": round(tot / len(QUESTIONS), 4)})
display(pd.DataFrame(sens).set_index("max_expand"))

### 怎么解释查询改写

比较 B3Q/B3Qs 与 B3，描述的是这套词表、模型、语料与题集上的结果，不能推出“查询改写只能用于 BM25”。
若两路扩展比只改稀疏路差，只能确认改变 dense 查询产生了影响；“偏离原始语义”仍需候选排序分析等证据。

上表在同一批 10 道题上扫描查询新增词数上限（`max_expand`），因此属于敏感性分析。保留默认 12，没有把最优值当作未见数据上的收益。B4 的一跳检索操作预算仍为 14，与查询新增词数分别配置。
修复后结果见下文与参考报告；旧版 0.700 / 0.731 等数字属于废弃协议，不作纵向对比。


In [ ]:
# 逐题看，才知道均值掩盖了什么
detail = []
for r in RESULT["per_question"]:
    detail.append({
        "id": r["id"], "层级": r["taxonomy"], "来源": r["source"],
        "B2": round(composite(r["B2"]), 3),
        "B3": round(composite(r["B3"]), 3),
        "B4": round(composite(r["B4"]), 3),
    })
df_detail = pd.DataFrame(detail).set_index("id")
display(df_detail)

worst = df_detail.assign(gap=df_detail["B4"] - df_detail["B3"]).nsmallest(3, "gap")
print("B4 相对 B3 退步最多的三题（这几题最值得去看 bad case）:")
display(worst)

In [ ]:
# 落盘。评测产物要能被别人重跑核对，不能只留一张截图
out = BASE / "results"
out.mkdir(exist_ok=True)
assert hashlib.sha256(QUESTIONS_PATH.read_bytes()).hexdigest() == QUESTIONS_SHA256, "Question set changed during evaluation"
report = {
    "provenance": PROVENANCE,
    "query_expansion_sensitivity": sens,
    "overall": df_overall.reset_index().to_dict("records"),
    "per_stratum": rows,
    "decision_checks": checks,
    "verdict": verdict,
    "passed": f"{passed}/{N_COMPARISONS}",
    "per_question": RESULT["per_question"],
}
(out / "h1_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False),
                                    encoding="utf-8")
print(f"已写入 {(out / 'h1_report.json').relative_to(BASE)}")

## 4.5 修复后的参考结果

<!-- RESULTS_START -->
| 臂 | Recall@5 | MRR@5 | nDCG@5 | composite |
|---|---|---|---|---|
| B2 | 0.542 | 0.633 | 0.477 | 0.551 |
| B3 | 0.612 | 0.717 | 0.555 | 0.628 |
| B4 | 0.637 | 0.750 | 0.577 | 0.655 |
| B3Q | 0.662 | 0.662 | 0.546 | 0.623 |
| B3Qs | 0.662 | 0.725 | 0.571 | 0.653 |
| B4Qs | 0.587 | 0.750 | 0.537 | 0.625 |

| 层级 | 题数 | B2 | B3 | B4 | B4−B2 | B4−B3 |
|---|---|---|---|---|---|---|
| L2 | 5 | 0.630 | 0.659 | 0.630 | -0.000 | -0.030 |
| L3 | 3 | 0.345 | 0.424 | 0.563 | 0.217 | 0.139 |

**四项通过 2/4，判定 `unsupported`。** 表格保留三位小数，判定使用未舍入值。

| 层级 | 两路改写−B3 | 仅稀疏改写−B3 | 扩展−B3 | 两者组合−B3 |
|---|---|---|---|---|
| L2 | 0.034 | 0.087 | -0.030 | -0.074 |
| L3 | -0.012 | -0.012 | 0.139 | 0.209 |
<!-- RESULTS_END -->

### 协议与解释边界

本次为 `strict-anchor-v2`：修正方法源码缩进解析、同名符号误匹配、依赖 gold 顺序的 nDCG；
统一 B4/B4Qs 的扩展策略，稳定同分排序，并使用未舍入均值判定。模块 docstring 的行坐标也改为实际 AST 范围。
这些是实现与评分协议修订，旧版数字已废弃，不能用新旧分数差宣称模型或 Agent 能力提升。

当前只测一跳证据扩展的排序。L2/L3 的差值是小题集上的描述性结果，不证明大型仓库中的通用优势。
本题集已用于开发，且 4 题由已有调用关系反推；没有独立 holdout、跨仓库验证或真实新人 A/B。

原课程项目的 33 题、495 次执行、三轮随机合成与 B3.5 消融，是另一套系统的作者报告数据；
本 notebook 没有复现它们，也不以那套历史结果证明这里的 ReAct 演示有效。

如果下一步接 LLM 最终回答评测，应统一各臂合成配置，记录候选与最终引用两套结果，
对随机路径做重复运行。确定性路径重复主要检查可复现性；扩题与重复的优先级取决于题目覆盖和运行波动。


---

本例提供两项可复用资产：精确锚点评分器和可替换检索路径的对照 harness。
将它迁移到其他项目时，先定义任务与 gold，再修改语料索引和对照臂，并重新验证所有锚点。
`reference/h1_report.json` 保留逐题结果、配置和代码哈希，`test_evaluation.py` 覆盖关键评分边界。

配套 Extra14 投稿讨论原课程项目与这份教学例子的关系；框架主包没有新增评测 API。
